# ACB Experiments — Free Replication on Google Colab

This notebook runs the three experiments (P1, P2, P3) from the paper
*The Agent Coordination Bound (ACB)* using **free GPU on Colab + Ollama**.

**Requirements:** Colab with GPU runtime (Runtime → Change runtime type → T4 GPU).

| Section | What it does | Needs GPU? | Time |
|---------|-------------|------------|------|
| 1. Install Ollama | System setup | No | 5-10 min |
| 2. Install ACB | Clone repo + deps | No | 2 min |
| 3. Download benchmarks | HumanEval + MATH | No | 3 min |
| 4. Validate math | Monte Carlo, CBI, figures | No | 30 sec |
| 5. Quick test | P1 only, 5 tasks | Yes | 10 min |
| 6. Full experiments | P1, P2, P3 | Yes | 24-48h |

---
## 1. Install Ollama

In [ ]:
# 1a: Install zstd (required by Ollama installer)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils > /dev/null 2>&1
print('\u2713 zstd installed')

In [ ]:
# 1b: Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh
print('\n\u2713 Ollama installed')

In [ ]:
# 1c: Start Ollama server
import subprocess, time, os
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama_stdout.log', 'w'),
    stderr=open('/tmp/ollama_stderr.log', 'w'),
)
print(f'Server started (PID {proc.pid}), waiting 10s...')
time.sleep(10)

import urllib.request
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5)
    print('\u2713 Ollama server is running')
except Exception as e:
    print(f'\u2717 Server error: {e}')
    !cat /tmp/ollama_stderr.log | tail -20

In [ ]:
# 1d: Pull model
MODEL = 'llama3.1:8b'  # For T4. Use llama3.1:70b on A100.
print(f'Pulling {MODEL}...')
!ollama pull {MODEL}
print(f'\n\u2713 {MODEL} ready')

In [ ]:
# 1e: Smoke test
!ollama run {MODEL} 'What is 2+2? Answer with just the number.' 2>&1 | head -3
print('\n\u2713 Model responds')

---
## 2. Install ACB

In [ ]:
import os
os.chdir('/content')

# Clone or detect existing repo
if not os.path.exists('/content/acb-experiments/acb/__init__.py'):
    # Maybe it's double-nested from tar extraction
    if os.path.exists('/content/acb-experiments/acb-experiments/acb/__init__.py'):
        print('Fixing double-nested folder...')
        !cp -r /content/acb-experiments/acb-experiments/* /content/acb-experiments/
        !rm -rf /content/acb-experiments/acb-experiments
    else:
        print('Cloning repository...')
        !git clone https://github.com/<YOUR-USERNAME>/acb-experiments.git

# Set working directory
os.chdir('/content/acb-experiments')
assert os.path.exists('acb/__init__.py'), 'ERROR: acb/ package not found!'
print(f'\u2713 Working directory: {os.getcwd()}')

# Install deps
!pip install numpy scipy pandas matplotlib seaborn pyyaml httpx python-dotenv tqdm datasets -q
print('\u2713 Dependencies installed')

In [ ]:
# Configure backend
import os, sys
os.environ['LOCAL_MODEL_URL'] = 'http://127.0.0.1:11434'
os.environ['LOCAL_MODEL_NAME'] = MODEL
os.environ['MAX_CONCURRENT'] = '1'
os.environ['SEED'] = '42'

with open('.env', 'w') as f:
    f.write(f'LOCAL_MODEL_URL=http://127.0.0.1:11434\n')
    f.write(f'LOCAL_MODEL_NAME={MODEL}\n')
    f.write(f'MAX_CONCURRENT=1\n')
    f.write(f'SEED=42\n')

# Ensure Python can find our modules
if '/content/acb-experiments' not in sys.path:
    sys.path.insert(0, '/content/acb-experiments')

print(f'\u2713 Backend: Ollama with {MODEL}')

---
## 3. Download Benchmarks

In [ ]:
# 3a: HumanEval (automatic via setup.py)
!PYTHONPATH=/content/acb-experiments python benchmarks/setup.py --humaneval

In [ ]:
# 3b: MATH dataset via HuggingFace (more reliable than git clone)
import json
from pathlib import Path

math_dir = Path('benchmarks/data/MATH/test')
if math_dir.exists() and any(math_dir.rglob('*.json')):
    n = len(list(math_dir.rglob('*.json')))
    print(f'\u2713 MATH already present ({n} tasks)')
else:
    from datasets import load_dataset
    print('Downloading MATH from HuggingFace...')
    ds = load_dataset('hendrycks/competition_math', split='test')

    counters = {}
    for item in ds:
        subject = item.get('type', 'unknown').lower().replace(' ', '_')
        subject_dir = math_dir / subject
        subject_dir.mkdir(parents=True, exist_ok=True)
        idx = counters.get(subject, 0)
        counters[subject] = idx + 1
        with open(subject_dir / f'{idx}.json', 'w') as f:
            json.dump({
                'problem': item['problem'],
                'solution': item['solution'],
                'level': item.get('level', ''),
                'type': item.get('type', ''),
            }, f)

    total = sum(counters.values())
    print(f'\u2713 MATH ready: {total} tasks across {len(counters)} subjects')
    for s, n in sorted(counters.items()):
        print(f'  {s}: {n}')

In [ ]:
# 3c: Verify everything is in place
!PYTHONPATH=/content/acb-experiments python benchmarks/setup.py --verify

---
## 4. Validate the Math (no GPU needed)

These run in seconds. They confirm the analytical formulas are correct.

In [ ]:
# 4a: Monte Carlo validation of P(harm|n) — reproduces Table 3
from monte_carlo.validate_pharm import run_validation
results = run_validation(mc_runs=50000)

In [ ]:
# 4b: CBI diagnostic — reproduces Table 6
from acb.cbi import interpret_cbi

for name, n, a, c in [
    ('AutoGen GroupChat (3 agents)', 3, 0.72, 0.082),
    ('LangChain 5-agent workflow',   5, 0.72, 0.082),
    ('AgentPrune BEFORE pruning',   20, 0.51, 0.065),
    ('AgentPrune AFTER pruning',     8, 0.51, 0.065),
    ('Self-consistency k=40',       40, 0.56, 0.041),
]:
    print(f'{name}:\n  {interpret_cbi(n, a, c)}\n')

In [ ]:
# 4c: Greedy fleet selection — reproduces Table 7
from acb.greedy_fleet import greedy_fleet_select

agents = [
    ('Claude-3.5-Sonnet', 0.89), ('GPT-4o', 0.87),
    ('GPT-4o-inst2', 0.86), ('GPT-4-Turbo', 0.82),
    ('Gemini-1.5-Pro', 0.79), ('GPT-4o-mini', 0.72),
    ('LLaMA-3-70B', 0.57),
]
result = greedy_fleet_select(agents, c=0.082, cross_model_penalty=1.3)
print('Greedy Heterogeneous Fleet Selection')
print('=' * 65)
for s in result.steps:
    stop = ' <- STOP' if s.stop else ''
    print(f'  Step {s.step}: {s.agent_name:25s} dI={s.marginal_gain:+.4f}  I={s.cumulative_I:.4f}{stop}')
print(f'\nSelected: {len(result.selected)} agents, I(S*) = {result.total_I:.4f}')

In [ ]:
# 4d: Patch plots.py if generate_all_figures is missing, then generate figures
import importlib
import analysis.plots
importlib.reload(analysis.plots)

if not hasattr(analysis.plots, 'generate_all_figures'):
    print('Patching plots.py with generate_all_figures...')
    patch = '''

def generate_all_figures(output_dir="figures/"):
    from pathlib import Path
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    print("Generating Figure 1A: Performance curves...")
    plot_performance_curves(
        configs=[
            ("GPT-4o-mini (a=0.72, c=0.082)", 0.72, 0.082),
            ("MATH debate (a=0.69, c=0.155)", 0.69, 0.155),
            ("Self-consistency (a=0.56, c=0.041)", 0.56, 0.041),
        ],
        output=str(out / "fig1a_performance_curves.png"),
    )
    print("Generating Figure 2A: Scaffold overhead...")
    plot_scaffold_overhead(output=str(out / "fig2a_scaffold_overhead.png"))
    print("Generating Figure 2B: CBI diagnostic...")
    plot_cbi_diagnostic(
        deployments=[
            ("AutoGen GroupChat", 3, 9), ("LangChain 5-agent", 5, 9),
            ("Society of Mind", 5, 18), ("AgentPrune (pre)", 20, 8),
            ("AgentPrune (post)", 8, 8), ("Self-consistency k=40", 40, 14),
        ],
        output=str(out / "fig2b_cbi_diagnostic.png"),
    )
    print("Generating Figure 3A: P(harm) validation...")
    plot_pharm_validation(
        mu_a=0.19, sigma_a=0.07, mu_c=0.18, mc_runs=50_000,
        output=str(out / "fig3a_pharm_validation.png"),
    )
    print(f"All figures saved to {output_dir}")
'''
    with open('analysis/plots.py', 'a') as f:
        f.write(patch)
    importlib.reload(analysis.plots)
    print('\u2713 Patched')

analysis.plots.generate_all_figures('figures/')

from IPython.display import Image, display
import glob
for fig in sorted(glob.glob('figures/*.png')):
    print(f'\n--- {os.path.basename(fig)} ---')
    display(Image(fig, width=600))

---
## 5. Quick Test (P1 only, ~10 min)

Run P1 with 5 tasks and 2 reps to verify the LLM pipeline works.

In [ ]:
!PYTHONPATH=/content/acb-experiments python run_all.py --quick --skip-mc --output-dir results/quick/

In [ ]:
# Check results
import json, glob
for f in sorted(glob.glob('results/quick/*.json')):
    data = json.load(open(f))
    print(f'\n{"=" * 60}')
    print(f'{data.get("experiment_name", "?")}')
    print(f'{"=" * 60}')
    for k, v in data.get('summary', {}).items():
        if k != 'llm_usage':
            print(f'  {k}: {v}')
    usage = data.get('summary', {}).get('llm_usage', {})
    if usage:
        print(f'  total_calls: {usage.get("total_calls", 0)}')
        print(f'  total_tokens: {usage.get("total_tokens", 0):,}')

---
## 6. Full Experiments (run individually)

Run each experiment separately. Save to Google Drive after each one
in case the Colab runtime disconnects.

| Experiment | Fleet sizes | Benchmark | Est. time (T4+8B) |
|-----------|-------------|-----------|-------------------|
| P1 | 1,2,3,4,5,6,7,8,9,10,12,15 | HumanEval (164) | 8-16h |
| P2 | 1,2,3,4,5,6,7,8,9,10,12,15 | MATH (200) | 12-24h |
| P3 | 1,3,6,9 | MATH (200) | 4-8h |

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/acb-results

In [ ]:
# P1: All-to-all fleet sizing on HumanEval
!PYTHONPATH=/content/acb-experiments python -m experiments.p1_fleet_sizing \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 \
    --reps 50 \
    --output-dir results/p1/

!cp -r results/p1/ /content/drive/MyDrive/acb-results/p1/ 2>/dev/null
print('\n\u2713 P1 complete')

In [ ]:
# P2: Supervisor vs all-to-all on MATH
!PYTHONPATH=/content/acb-experiments python -m experiments.p2_topology_crossover \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 \
    --reps 50 \
    --max-tasks 200 \
    --output-dir results/p2/

!cp -r results/p2/ /content/drive/MyDrive/acb-results/p2/ 2>/dev/null
print('\n\u2713 P2 complete')

In [ ]:
# P3: Shared vs isolated RAG on MATH
!PYTHONPATH=/content/acb-experiments python -m experiments.p3_rag_diversity \
    --fleet-sizes 1 3 6 9 \
    --reps 50 \
    --max-tasks 200 \
    --output-dir results/p3/

!cp -r results/p3/ /content/drive/MyDrive/acb-results/p3/ 2>/dev/null
print('\n\u2713 P3 complete')

---
## 7. Analyze Results

In [ ]:
import json, glob, os

for f in sorted(glob.glob('results/**/*.json', recursive=True)):
    data = json.load(open(f))
    s = data.get('summary', {})
    print(f'\n{"=" * 60}')
    print(f'{data.get("experiment_name", "?")}  ({os.path.basename(f)})')
    print(f'{"=" * 60}')
    for k, v in s.items():
        if k != 'llm_usage':
            print(f'  {k}: {v}')

In [ ]:
# P1 verdict
p1_files = sorted(glob.glob('results/p1/*.json') + glob.glob('results/quick/p1*.json'))
if p1_files:
    s = json.load(open(p1_files[-1]))['summary']
    print('P1: PERFORMANCE PEAK PREDICTION')
    print(f'  a = {s["a"]:.4f}')
    print(f'  c = {s["c"]:.6f}')
    print(f'  n* predicted  = {s["n_star"]}')
    print(f'  Empirical peak = {s["empirical_peak"]}')
    match = 'CONFIRMED' if s['n_star_matches'] else 'FALSIFIED'
    print(f'  Verdict: {match}')
    print(f'\n  Pass@1 by fleet size:')
    for n, acc in sorted(s['pass_at_1'].items(), key=lambda x: int(x[0])):
        bar = chr(9608) * int(acc * 40)
        print(f'    n={int(n):2d}: {acc:.3f} {bar}')
else:
    print('No P1 results found.')

In [ ]:
# P2 verdict
p2_files = sorted(glob.glob('results/p2/*.json'))
if p2_files:
    s = json.load(open(p2_files[-1]))['summary']
    print('P2: TOPOLOGY CROSSOVER')
    print(f'  Confirmed: {s.get("p2_confirmed", "N/A")}')
    print(f'  Falsified: {s.get("p2_falsified", "N/A")}')
    for n, stats in sorted(s.get('comparisons', {}).items(), key=lambda x: int(x[0])):
        w = 'SUP' if stats['sup_wins'] else 'A2A'
        sig = '*' if stats.get('significant') else ''
        print(f'    n={int(n):2d}: a2a={stats["a2a_acc"]:.3f} sup={stats["sup_acc"]:.3f} -> {w}{sig}')
else:
    print('No P2 results found.')

In [ ]:
# P3 verdict
p3_files = sorted(glob.glob('results/p3/*.json'))
if p3_files:
    s = json.load(open(p3_files[-1]))['summary']
    print('P3: CONTEXT DIVERSITY')
    print(f'  Baseline (n=1): {s.get("baseline_acc", "N/A")}')
    print(f'  rho_crit: {s.get("rho_crit", "N/A")}')
    print(f'  P3 confirmed: {s.get("p3_confirmed", "N/A")}')
    for n, stats in sorted(s.get('by_fleet_size', {}).items(), key=lambda x: int(x[0])):
        print(f'    n={int(n):2d}: shared={stats["shared_acc"]:.3f} isolated={stats["isolated_acc"]:.3f}')
else:
    print('No P3 results found.')